# opencellid_senegal_90d dataset
-  opencellid dataset for Senegal with 90 days observation window

- Source:
> [Cell Towers Worldwide: Location Data by Continent](
https://www.kaggle.com/datasets/zakariaeyoussefi/cell-towers-worldwide-location-data-by-continent)
>[https://www.kaggle.com/datasets/zakariaeyoussefi/cell-towers-worldwide-location-data-by-continent](https://www.kaggle.com/datasets/zakariaeyoussefi/cell-towers-worldwide-location-data-by-continent)

This extensive dataset provides geographic coordinates and network information for cell tower locations across the globe, organized by continent. It includes the following columns:

- Radio: The generation of broadband cellular network technology (e.g., LTE, GSM).
- MCC: Mobile Country Code, a unique identifier for each country in the mobile network.
- MNC: Mobile Network Code, identifying the mobile network within a country.
- LAC: Location Area Code, Tracking Area Code, or Network Identifier.
- CID: Unique identifier for each Base Transceiver Station (BTS) or sector.
- Longitude: Geographic coordinate specifying the east-west position.
- Latitude: Geographic coordinate specifying the north-south position.
- Range: Approximate area within which the cell coverage extends (in meters).
- Samples: Number of measures processed to derive the data point.
- Changeable: Indicates if the cell location was determined through sample processing (1) or directly obtained from the telecom firm (0).
- Created: Timestamp indicating when the cell was first added to the database (UNIX format).
- Updated: Timestamp indicating when the cell was last seen or updated in the database (UNIX format).
- AverageSignal: Represents the averaged signal strength of the cell location.
- Country: The country of the cell tower.
- Network: The company that owns the cell tower.
- Continent: The continent of the cell tower.


### Filtering
- Filter Senegal data from Africa (mcc=608)
  - Mobile Country Code (MCC) for Senegal is 608

- Filter Express from Orange and Tiggo (net=2)
  - Mobile Network Code (MNC) for Expresso is 02
  
### 90 days observation window
- Consider 90 days to match the churn definition
  - churn indicates whether a customer becomes inactive and makes no transactions for 90 consecutive days.

### Based on [OpenCelliD](https://opencellid.org/)

# Step 1 Mount Google Drive

**Authorize access when prompted.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


# Step 2  Install and import libraries
- Core  
  - os:managing  file paths, environment variables, and directory structures
  - numpy: computing multi-dimensional array operations.
  - pandas: manipulating structured data

- Scaling
  - dask: data processing by distributing large datasets across multiple partitions and executing computations in parallel
    - dask.diagnostics.ProgressBar: visual rendering of the progress of parallel computations, helping you monitor long-running Dask tasks
  - pyarrow: providing a high-performance interface for in-memory columnar data using for Apache Arrow



In [ ]:
# Uncomment on first run in a fresh Colab session
# !pip install numpy pandas  "dask[complete]"

In [ ]:
import os

import numpy as np
import pandas as pd
import dask.dataframe as dd
from dask.diagnostics import ProgressBar

# view all columns in a pandas DataFrame
pd.set_option("display.max_columns", None)

# Step 3 Define File Paths




In [ ]:
# Base Path
# BASE = "/path/to/datasets"

# OpenCellID Data
# opencellid_file              = f"{BASE}/opencellid/Africa_towers.csv"

# Processed Data
# opencellid_senegal_90d_file = f"{BASE}/opencellid/opencellid_senegal_90d.csv"

# Step 4 Load Africa_towers


In [ ]:
# Define datatypes
dtypes = {
    "radio": "category",
    "MCC": "int16",
    "MNC": "int8",
    "TAC": "int32",
    "CID": "int64",
    "unit": "int16",
    "LON": "float32",
    "LAT": "float32",
    "RANGE": "float32",
    "SAM": "int16",
    "changeable": "int8",
    "created": "int64",
    "updated": "int64",
    "averageSignal": "float32",
    "Country": "category",
    "Network": "category",
    "Continent": "category"
}

# Load only needed columns
columns = list(dtypes.keys())
columns

['radio',
 'MCC',
 'MNC',
 'TAC',
 'CID',
 'unit',
 'LON',
 'LAT',
 'RANGE',
 'SAM',
 'changeable',
 'created',
 'updated',
 'averageSignal',
 'Country',
 'Network',
 'Continent']

In [ ]:
print("\nLoading Africa_towers dataset with Dask …")
opencellid = dd.read_csv(
    opencellid_file,
    usecols=columns,
    dtype=dtypes,
    blocksize="32MB"   # smaller partitions for safer RAM usage
)

print(f"Dask partitions : {opencellid.npartitions}")
print(f"Column names    : {list(opencellid.columns)}")
print(f"Number of rows  : {len(opencellid)}")


Loading Africa_towers dataset with Dask …
Dask partitions : 8
Column names    : ['radio', 'MCC', 'MNC', 'TAC', 'CID', 'unit', 'LON', 'LAT', 'RANGE', 'SAM', 'changeable', 'created', 'updated', 'averageSignal', 'Country', 'Network', 'Continent']
Number of rows  : 2346316


In [ ]:
print(f"Preview first rows of first partition        : \n{opencellid.head()}")
print(f"Preview last rows of last partition       : \n{opencellid.tail()}")

Preview first rows of first partition        : 
  radio  MCC  MNC    TAC    CID  unit        LON        LAT    RANGE  SAM  \
0   GSM  602    3  21333  25372     0  31.056511  29.998215  23897.0   10   
1   GSM  602    3  21362  23224     0  31.373520  29.839554   1000.0    4   
2   GSM  602    3  22533   5031     0  31.160660  29.998856   1000.0    1   
3   GSM  602    3  22202  40686     0  31.501236  30.592117   1000.0    1   
4   GSM  602    3  21333  25376     0  31.277390  30.095673   1000.0    2   

   changeable     created     updated  averageSignal Country   Network  \
0           1  1459715136  1465348064            0.0   Egypt  Etisalat   
1           1  1459813831  1474212997            0.0   Egypt  Etisalat   
2           1  1459695955  1459695955            0.0   Egypt  Etisalat   
3           1  1459681907  1459681907            0.0   Egypt  Etisalat   
4           1  1459715136  1460478702            0.0   Egypt  Etisalat   

  Continent  
0    Africa  
1    Africa  
2 

### Check Memory Usage Before Filtering

In [ ]:
print(opencellid.memory_usage(deep=True).compute().sum() / 1e6, "MB")

131.418504 MB


# Step 5 Filter and persist  data

- filtering Senegal and expresso data from:
>[International Telecommunication Union (2023). Mobile Network Codes (MNC) for the international identification plan for public networks and subscriptions](https://www.itu.int/dms_pub/itu-t/opb/sp/T-SP-E.212B-2023-PDF-E.pdf)

### Filtering Senegal data:
- Filter Senegal Towers using Mobile Country Code (MCC) (MCC = 608)

### Filtering expresso data:
- Filter Express Towers from other operators Mobile Network Code (MNC) (MNC=3)

### .persist()
- .persist() triggers the computation of the current task graph and keeps the results in memory as a distributed Dask collection for faster future access.
- persist filtered data early to prevent recomputation.

In [ ]:
# Combine the two filters
opencellid = opencellid[(opencellid["MCC"] == 608) & (opencellid["MNC"] == 3)].persist()

In [ ]:
# Verify the result
print(f"Filtered Rows (Expresso Senegal): {len(opencellid)}")

Filtered Rows (Expresso Senegal): 3272



### Check Memory Usage After Filtering

In [ ]:
print(opencellid.memory_usage(deep=True).compute().sum() / 1e6, "MB")

0.23316 MB


# Step 6 Compute Dask Graph → pandas DataFrame

- trigger the actual execution of all queued Dask operations to convert the distributed dataframe into a standard, in-memory Pandas dataframe


In [ ]:
print("\nComputing … (2–5 min on standard Colab)")
with ProgressBar():
    opencellid = opencellid.compute()

# Clean up the Categories
opencellid["Country"] = opencellid["Country"].cat.remove_unused_categories()
opencellid["Network"] = opencellid["Network"].cat.remove_unused_categories()
opencellid["Continent"] = opencellid["Continent"].cat.remove_unused_categories()

# Fix repeated partition indexes
opencellid = opencellid.reset_index(drop=True)

print(f"Computed shape : {opencellid.shape}")
print(f"Index range    : {opencellid.index.min()} → {opencellid.index.max()}")
print(f"Index unique   : {opencellid.index.is_unique}")


Computing … (2–5 min on standard Colab)
[########################################] | 100% Completed | 104.20 ms
Computed shape : (3272, 17)
Index range    : 0 → 3271
Index unique   : True


In [ ]:
opencellid.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3272 entries, 0 to 3271
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   radio          3272 non-null   category
 1   MCC            3272 non-null   int16   
 2   MNC            3272 non-null   int8    
 3   TAC            3272 non-null   int32   
 4   CID            3272 non-null   int64   
 5   unit           3272 non-null   int16   
 6   LON            3272 non-null   float32 
 7   LAT            3272 non-null   float32 
 8   RANGE          3272 non-null   float32 
 9   SAM            3272 non-null   int16   
 10  changeable     3272 non-null   int8    
 11  created        3272 non-null   int64   
 12  updated        3272 non-null   int64   
 13  averageSignal  3272 non-null   float32 
 14  Country        3272 non-null   category
 15  Network        3272 non-null   category
 16  Continent      3272 non-null   category
dtypes: category(4), float32(4), int16

In [ ]:
print(f"Preview first rows       : \n{opencellid.head()}")
print(f"Preview last rows        : \n{opencellid.tail()}")

Preview first rows       : 
  radio  MCC  MNC  TAC    CID  unit        LON        LAT    RANGE  SAM  \
0   GSM  608    3  801  60053     0 -14.943466  12.890396   1000.0    5   
1   GSM  608    3  401  43013     0 -16.500776  14.361879   2792.0    2   
2   GSM  608    3  401  43011     0 -16.496656  14.361879   3218.0    2   
3   GSM  608    3  701  57042     0 -14.722878  13.998784  56823.0    5   
4   GSM  608    3  701  57045     0 -14.593620  13.982508   3757.0    4   

   changeable     created     updated  averageSignal  Country   Network  \
0           1  1459813076  1460884030            0.0  Senegal  Expresso   
1           1  1351194533  1456970224            0.0  Senegal  Expresso   
2           1  1351194533  1456970224            0.0  Senegal  Expresso   
3           1  1351272426  1478774472            0.0  Senegal  Expresso   
4           1  1351272426  1472278594            0.0  Senegal  Expresso   

  Continent  
0    Africa  
1    Africa  
2    Africa  
3    Africa  


# Step 7 Validate Filtering

- Verify that the filters worked correctly.

In [ ]:
opencellid["Country"].value_counts()

,count
Country,
Senegal,3272


In [ ]:
opencellid["Network"].value_counts()

,count
Network,
Expresso,3272


# Step 8 Convert UNIX Timestamps to Datetime

- the dataset uses UNIX timestamps.

- convert raw UNIX epoch seconds into human-readable datetime objects to perform time-series analysis and filtering on towers data.

- the "unit='s'" parameter indicates that the large numbers (e.g., 1459715000) represent seconds elapsed since January 1, 1970.



In [ ]:
opencellid["created_date"] = pd.to_datetime(opencellid["created"], unit="s")
opencellid["updated_date"] = pd.to_datetime(opencellid["updated"], unit="s")

In [ ]:
opencellid.head()

,radio,MCC,MNC,TAC,CID,unit,LON,LAT,RANGE,SAM,changeable,created,updated,averageSignal,Country,Network,Continent,created_date,updated_date
0,GSM,608,3,801,60053,0,-14.943466,12.890396,1000.0,5,1,1459813076,1460884030,0.0,Senegal,Expresso,Africa,2016-04-04 23:37:56,2016-04-17 09:07:10
1,GSM,608,3,401,43013,0,-16.500776,14.361879,2792.0,2,1,1351194533,1456970224,0.0,Senegal,Expresso,Africa,2012-10-25 19:48:53,2016-03-03 01:57:04
2,GSM,608,3,401,43011,0,-16.496656,14.361879,3218.0,2,1,1351194533,1456970224,0.0,Senegal,Expresso,Africa,2012-10-25 19:48:53,2016-03-03 01:57:04
3,GSM,608,3,701,57042,0,-14.722878,13.998784,56823.0,5,1,1351272426,1478774472,0.0,Senegal,Expresso,Africa,2012-10-26 17:27:06,2016-11-10 10:41:12
4,GSM,608,3,701,57045,0,-14.593620,13.982508,3757.0,4,1,1351272426,1472278594,0.0,Senegal,Expresso,Africa,2012-10-26 17:27:06,2016-08-27 06:16:34


# Step 9 90-day observation window

- Choosing the 90-day observation window depends on the Expresso dataset from the Expresso Churn Prediction Challenge

- the official documentation for Expresso Churn Prediction Challenge on Zindi, does not provide a specific "calendar" date (like January 1st to March 30th) because the data is anonymized to protect customer privacy. Instead, the timeframe is defined by a rolling 90-day observation window.

- "The objective of this challenge is to predict the likelihood of each customer 'churning,' i.e. becoming inactive and not making any transactions for 90 days."
>[Zindi (2021). Expresso Churn Prediction Challenge](https://zindi.africa/competitions/expresso-churn-prediction/data)

- Given that the challenge was started on 27 August 2021, but the specific dataset dates were not provided, we have defined the 90-day observation period as May – August 2021. This selection is based on the Zindi competition’s 90-day churn definition and serves to contextualize behavioral patterns within a realistic three-month timeframe."
>[Zindi (2021). Expresso Churn Prediction Challenge, Discussions: Date & Time ](
https://zindi.africa/competitions/expresso-churn-prediction/discussions/7603)



In [ ]:
print(f"Min created_date: {opencellid["created_date"].min()}")
print(f"Max created_date: {opencellid["created_date"].max()}")

Min created_date: 2012-10-25 19:48:53
Max created_date: 2023-09-15 14:25:11


In [ ]:
opencellid["created_date"].dt.year.value_counts().sort_index()

,count
created_date,
2012,4
2013,146
2014,83
2015,904
2016,1836
2017,173
2018,105
2019,16
2023,5


In [ ]:
print(f"Min updated_date: {opencellid["updated_date"].min()}")
print(f"Max updated_date: {opencellid["updated_date"].max()}")

Min updated_date: 2013-02-25 12:22:25
Max updated_date: 2023-09-15 14:25:11


In [ ]:
opencellid["updated_date"].dt.year.value_counts().sort_index()

,count
updated_date,
2013,62
2014,7
2015,612
2016,1566
2017,893
2018,111
2019,16
2023,5


# Step 10 Select 90 Days of Network Data Snapshot

- the network dataset is filtered to include only those towers active within the designated observation window to maintain consistency with the 90-day churn definition (where churn is defined by three months of inactivity)

- By establishing a **start_date** and a **end_date**, we ensure that the spatial features used in the model reflect the actual network state during the customer activity snapshot.

> Ensure the tower data aligns with the churn observation window (May - August 2021)

- churn definition is:

> A customer is considered churned if they are inactive for 90 consecutive days.

- Therefore, the network features must represent the network conditions during that same 90-day observation period.

- churn observation window:

> May 1, 2021  →  August 1, 2021

- include both created_date and updated_date

- ensure the tower was active or observed during that period.

- a tower may have been created earlier but still active in the window.

- keep towers that:

  - existed before August
  - were still observed during the 90-day window


In [ ]:
# Define the churn observation window
start_date = pd.to_datetime("2019-04-01")
end_date   = pd.to_datetime("2019-07-01")

print(f"start_date: {start_date}")
print(f"end_date: {end_date}")

start_date: 2019-04-01 00:00:00
end_date: 2019-07-01 00:00:00


In [ ]:
opencellid_90d.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3267 entries, 0 to 3266
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   radio          3267 non-null   category      
 1   MCC            3267 non-null   int16         
 2   MNC            3267 non-null   int8          
 3   TAC            3267 non-null   int32         
 4   CID            3267 non-null   int64         
 5   unit           3267 non-null   int16         
 6   LON            3267 non-null   float32       
 7   LAT            3267 non-null   float32       
 8   RANGE          3267 non-null   float32       
 9   SAM            3267 non-null   int16         
 10  changeable     3267 non-null   int8          
 11  created        3267 non-null   int64         
 12  updated        3267 non-null   int64         
 13  averageSignal  3267 non-null   float32       
 14  Country        3267 non-null   category      
 15  Network        3267 non-nu

In [ ]:
opencellid_90d.head()

,radio,MCC,MNC,TAC,CID,unit,LON,LAT,RANGE,SAM,changeable,created,updated,averageSignal,Country,Network,Continent,created_date,updated_date
0,GSM,608,3,801,60053,0,-14.943466,12.890396,1000.0,5,1,1459813076,1460884030,0.0,Senegal,Expresso,Africa,2016-04-04 23:37:56,2016-04-17 09:07:10
1,GSM,608,3,401,43013,0,-16.500776,14.361879,2792.0,2,1,1351194533,1456970224,0.0,Senegal,Expresso,Africa,2012-10-25 19:48:53,2016-03-03 01:57:04
2,GSM,608,3,401,43011,0,-16.496656,14.361879,3218.0,2,1,1351194533,1456970224,0.0,Senegal,Expresso,Africa,2012-10-25 19:48:53,2016-03-03 01:57:04
3,GSM,608,3,701,57042,0,-14.722878,13.998784,56823.0,5,1,1351272426,1478774472,0.0,Senegal,Expresso,Africa,2012-10-26 17:27:06,2016-11-10 10:41:12
4,GSM,608,3,701,57045,0,-14.593620,13.982508,3757.0,4,1,1351272426,1472278594,0.0,Senegal,Expresso,Africa,2012-10-26 17:27:06,2016-08-27 06:16:34


In [ ]:
# Validate the Result
print(opencellid_90d.shape)

(3267, 19)


In [ ]:
print(f"Min created_date: {opencellid_90d["created_date"].min()}")
print(f"Max created_date: {opencellid_90d["created_date"].max()}")

Min created_date: 2012-10-25 19:48:53
Max created_date: 2019-05-25 14:18:10


In [ ]:
print(f"Min updated_date: {opencellid_90d["updated_date"].min()}")
print(f"Max updated_date: {opencellid_90d["updated_date"].max()}")

Min updated_date: 2013-02-25 12:22:25
Max updated_date: 2019-05-31 18:07:06


# Step 11 Save opencellid_senegal_90d


In [ ]:
print(f"Saving …")
os.makedirs(os.path.dirname(opencellid_senegal_90d_file), exist_ok=True)

opencellid_90d.to_csv(opencellid_senegal_90d_file, index=False)

size_mb = os.path.getsize(opencellid_senegal_90d_file) / 1024 / 1024
print(f"Saved — {len(opencellid_90d):,} rows | {size_mb:.1f} MB")


Saving …
Saved — 3,267 rows | 0.5 MB
